## Setup

In [2]:
from google.colab import userdata
access_token = userdata.get('CASM-NER')

In [3]:
%%capture
!pip install transformers
!pip install sentencepiece
!pip install seqeval
!pip install datasets
# !pip install git+https://github.com/ay94/multilingual-ner.git

In [ ]:
# from ner import evaluation

In [4]:
## Mount GDrive
from google.colab import drive
drive.mount('/content/drive/', force_remount=True)

## Imports
import os
import sys
import nltk
import time
import torch
import random
import subprocess
import numpy as np
import pandas as pd
import datetime as dt
from itertools import groupby
from tqdm.notebook import tqdm
from datasets import load_dataset
from transformers import pipeline
from collections import Counter, defaultdict
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForTokenClassification, AutoTokenizer
from seqeval.metrics import f1_score as seq_f1, precision_score as seq_precision, recall_score as seq_recall, classification_report as seq_classification
from sklearn.metrics import f1_score as skl_f1, precision_score as skl_precision, recall_score as skl_recall, classification_report as skl_classification

Mounted at /content/drive/


In [5]:
# Append the library files into the notebook system path for import
sys.path.append('/content/drive/Shareddrives/Machine Translation/Model benchmarking/Libraries/1.0.2')
# import custom library files
import ner, utils

## Load datasets

### masakhane/masakhaner2

In [14]:
label_map = {
    "O": 0,
    "B-PER": 1,
    "I-PER": 2,
    "B-ORG": 3,
    "I-ORG": 4,
    "B-LOC": 5,
    "I-LOC": 6,
    "B-DATE": 7,
    "I-DATE": 8,
}

masakhaner2 = ner.ReadNERData()
masakhaner2_words, masakhaner2_labels = masakhaner2.read_dataset('masakhane/masakhaner2', label_map, lang='zul')

Generating test Split


  0%|          | 0/1670 [00:00<?, ?it/s]

In [15]:
print(ner.check_labels(masakhaner2_labels))
label_alignment = {
    'I-PER': 'I-PER',
    'I-DATE': 'O',
    'B-ORG': 'B-ORG',
    'B-LOC': 'B-LOC',
    'I-LOC': 'I-LOC',
    'O':     'O',
    'B-DATE': 'O',
    'B-PER': 'B-PER',
    'I-ORG': 'I-ORG',
}

# Align the dataset labels to the standard labels
masakhaner2_labels = ner.align_dataset(masakhaner2_labels, label_alignment)
print(ner.check_labels(masakhaner2_labels))
# Dataset Label Map Alignment to LOC, ORG, PERS, MISC

{'I-PER', 'I-DATE', 'B-ORG', 'B-LOC', 'I-LOC', 'O', 'B-DATE', 'B-PER', 'I-ORG'}
{'I-PER', 'B-ORG', 'B-LOC', 'I-LOC', 'O', 'B-PER', 'I-ORG'}


# Evaluate model

In [12]:
alignment = {
'B-organization': 'B-ORG',
'O': 'O',
'B-other': 'O',
'B-person': 'B-PER',
'I-person': 'I-PER',
'B-location': 'B-LOC',
'I-organization': 'I-ORG',
'I-other': 'O',
'I-location': 'I-LOC'
}

model_name = "tner/xlm-roberta-large-conll2003"
model_name_output = 'tner-xlm-roberta-large'
model_evaluation = ner.ModelEvaluation(
    model_name,
    alignment
)

tokenizer_config.json:   0%|          | 0.00/212 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.01k [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

In [13]:
# model_evaluation.model.config.id2label

### masakhane/masakhaner2

In [16]:
data_name = "masakhane/masakhaner2"
masakhaner2_evaluation_output = model_evaluation.evaluate_model(masakhaner2_words, masakhaner2_labels)

  0%|          | 0/105 [00:00<?, ?it/s]

In [17]:
masakhaner2_seqeval = masakhaner2_evaluation_output.get_classification('Seqeval')
masakhaner2_seqeval

,Tag,Precision,Recall,F1,support
0,LOC,0.4802,0.2522,0.3307,337
1,ORG,0.3800,0.3566,0.3679,373
2,PER,0.5727,0.3727,0.4516,888
3,micro,0.4968,0.3436,0.4062,1598
4,macro,0.4776,0.3272,0.3834,1598
5,weighted,0.5082,0.3436,0.4066,1598


In [18]:
masakhaner2_sklearn = masakhaner2_evaluation_output.get_classification('Sklearn')
masakhaner2_sklearn

,Tag,Precision,Recall,F1,support
0,B-LOC,0.7143,0.2671,0.3888,337
1,B-ORG,0.5018,0.3673,0.4241,373
2,B-PER,0.7311,0.3491,0.4726,888
3,I-LOC,0.8146,0.7987,0.8066,154
4,I-ORG,0.6106,0.4829,0.5393,263
5,I-PER,0.8468,0.8103,0.8282,464
6,O,0.9545,0.9889,0.9714,23607
7,accuracy,0.9395,26086,None,None
8,macro,0.7391,0.5806,0.6330,26086
9,weighted,0.9311,0.9395,0.9312,26086
